# 📄 Pipeline de Reconstruction de Structure Documentaire
## Architecture Text-First & OCR Ciblé Parallèle à Deux Niveaux

---

> **Objectif Réel :**
> Reconstruire une **partition structurelle robuste** du document PDF en régions continues de pages (`start_page` → `end_page`),
> avec 100 % de couverture de pages, zéro suppression de page scannée, zéro faux heading,
> et élimination du gaspillage de calcul par **OCR ciblé uniquement sur les zones d'incertitude**.

### 🏗️ Architecture du Pipeline Optimisé

```
                         PDF
                          │
                          ▼
                PAGE CLASSIFICATION
                          │
             ┌────────────┴────────────┐
             │                         │
            TEXT                    SCANNED
             │                         │
             ▼                         ▼
      Native extraction         (Conservée intacte)
             │                         │
             ▼                         │
     TOC & BOOKMARKS                   │
             │                         │
             ▼                         │
   NATIVE CANDIDATES & ANCHORS         │
             │                         │
             └────────────┬────────────┘
                          ▼
             IDENTIFY OCR REQUIRED PAGES
        (TOC target, sequence gaps, boundaries)
                          │
             ┌────────────┴────────────┐
             ▼                         ▼
      OCR REQUIRED / OPTIONAL    OCR NOT NEEDED
             │                   (Milieu de section)
             ▼                         │
    PARALLEL TARGETED OCR              │
    ├─ Level 1: Light (Top 30%)        │
    └─ Level 2: Full (si candidat)     │
    (Cache mémoire & .ocr_cache/)       │
             │                         │
             └────────────┬────────────┘
                          ▼
            FINAL ANCHOR VALIDATION
                          │
                          ▼
             HIERARCHY & SUB-DOCUMENTS
                          │
                          ▼
            SECTION BOUNDARIES (start/end)
                          │
                          ▼
         CONTINUOUS 100% PAGE ASSIGNMENT
                          │
                          ▼
         10-RULE STRUCTURAL VALIDATION
                          │
                          ▼
    structure.json + page_map.json + structure_debug.json
```


---
## ⚙️ CELL 1 — Configuration

Définition des chemins d'entrée/sortie, des seuils de classification, des paramètres d'OCR ciblé, du nombre de workers et du cache.


In [1]:
# =========================================================================
# CELL 1 — Configuration
# =========================================================================

import os
import sys
from pathlib import Path

# --- ENTRÉES / SORTIES ---
DATA_DIR   = Path("data")
INPUT_PDF  = DATA_DIR / "MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf"

if not INPUT_PDF.exists() and (DATA_DIR / "MYCOM.pdf").exists():
    INPUT_PDF = DATA_DIR / "MYCOM.pdf"
elif not INPUT_PDF.exists() and (DATA_DIR / "AUSTCOLD.pdf").exists():
    INPUT_PDF = DATA_DIR / "AUSTCOLD.pdf"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- PARAMÈTRES PIPELINE & OCR CIBLÉ ---
TEXT_CHAR_THRESHOLD           = 20     # Pages avec < 20 caractères classées SCANNED
ENABLE_OCR                    = True   # Activer l'OCR ciblé
OCR_ONLY_WHEN_NEEDED          = True   # N'OCRise que les zones d'incertitude réelles
OCR_LIGHT_MODE                = True   # Niveau 1 : Light OCR (top band)
OCR_TOP_RATIO                 = 0.30   # Région haute analysée (top 30%)
OCR_FULL_ON_HEADING_CANDIDATE = True   # Niveau 2 : Full OCR si candidat détecté
MAX_OCR_WORKERS               = min(4, os.cpu_count() or 4)
OCR_CACHE_ENABLED             = True   # Cache mémoire et disque
OCR_CACHE_DIR                 = ".ocr_cache"
OCR_LOCAL_WINDOW              = 5      # Fenêtre locale (±5 pages)
OCR_EXPANDED_WINDOW           = 15     # Fenêtre étendue (±15 pages)
MIN_HEADING_CONFIDENCE        = 0.55   # Seuil minimal pour valider un anchor

print(f"[OK] Fichier cible           : {INPUT_PDF.resolve()}")
print(f"[OK] Répertoire out          : {OUTPUT_DIR.resolve()}")
print(f"[OK] OCR ciblé activé        : {ENABLE_OCR} (Workers: {MAX_OCR_WORKERS})")
print(f"[OK] Mode OCR                : {'Light Top ' + str(int(OCR_TOP_RATIO*100)) + '% -> Full' if OCR_LIGHT_MODE else 'Full'}")
print(f"[OK] Cache disque activé     : {OCR_CACHE_ENABLED} ({OCR_CACHE_DIR})")


[OK] Fichier cible           : C:\Users\camelia\Downloads\projets\projet1OCP\data\MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf
[OK] Répertoire out          : C:\Users\camelia\Downloads\projets\projet1OCP\output
[OK] OCR ciblé activé        : True (Workers: 4)
[OK] Mode OCR                : Light Top 30% -> Full
[OK] Cache disque activé     : True (.ocr_cache)


---
## 📂 CELL 2 — Load PDF

Ouverture du fichier PDF via PyMuPDF (`fitz`), extraction des métadonnées initiales et vérification des signets natifs (`get_toc()`).


In [2]:
# =========================================================================
# CELL 2 — Load PDF
# =========================================================================

import fitz

if not INPUT_PDF.exists():
    raise FileNotFoundError(f"Fichier PDF introuvable : {INPUT_PDF}")

doc = fitz.open(str(INPUT_PDF))
total_pages = len(doc)
raw_toc = doc.get_toc(simple=True)

print(f"[OK] Document chargé        : {INPUT_PDF.name}")
print(f"     Pages physiques totales: {total_pages}")
print(f"     Signets natifs (TOC)   : {len(raw_toc)} entrée(s)")


[OK] Document chargé        : MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf
     Pages physiques totales: 1068
     Signets natifs (TOC)   : 0 entrée(s)


---
## 🏷️ CELL 3 — Page Classification (TEXT / SCANNED / MIXED)

Passe ultra-rapide d'ingestion et de classification de toutes les pages physiques.
- **TEXT** : Contenu natif suffisant (≥ 20 caractères).
- **SCANNED** : Image scannée sans texte natif. *Contrainte absolue : aucune page scannée n'est supprimée.*
- **MIXED** : Texte natif + diagrammes / images.


In [3]:
# =========================================================================
# CELL 3 — Page Classification
# =========================================================================

import time
from document_structure import extract_and_classify_pages

t0 = time.time()
pages_data = extract_and_classify_pages(
    doc,
    text_char_threshold=TEXT_CHAR_THRESHOLD
)
t_class = time.time() - t0

text_count    = sum(1 for p in pages_data if p["page_type"] == "TEXT")
scanned_count = sum(1 for p in pages_data if p["page_type"] == "SCANNED")
mixed_count   = sum(1 for p in pages_data if p["page_type"] == "MIXED")
paginated     = sum(1 for p in pages_data if p.get("internal_page") is not None)

print(f"[OK] Classification terminée en {t_class:.2f}s pour {total_pages} pages")
print(f"     TEXT    : {text_count:>5} pages ({text_count/total_pages*100:.1f}%)")
print(f"     SCANNED : {scanned_count:>5} pages ({scanned_count/total_pages*100:.1f}%) [Préservées à 100%]")
print(f"     MIXED   : {mixed_count:>5} pages ({mixed_count/total_pages*100:.1f}%)")
print(f"     Pagination interne détectée : {paginated} pages")


[OK] Classification terminée en 4.05s pour 1068 pages
     TEXT    :   821 pages (76.9%)
     SCANNED :   167 pages (15.6%) [Préservées à 100%]
     MIXED   :    80 pages (7.5%)
     Pagination interne détectée : 49 pages


---
## 📑 CELL 4 — Extract Bookmarks & Visible TOC

Extraction des deux sources externes de structure :
1. **Source A : Bookmarks natifs** (`get_toc()`).
2. **Source B : Sommaire visible (CONTENTS / SOMMAIRE)** dans les pages du document.


In [4]:
# =========================================================================
# CELL 4 — Extract Bookmarks & Visible TOC
# =========================================================================

from document_structure import extract_bookmarks, detect_contents_pages, extract_contents_entries

# Source A: Bookmarks natifs
bookmarks = extract_bookmarks(doc)
print(f"[Source A] Bookmarks natifs : {len(bookmarks)} entrée(s)")
if bookmarks:
    for b in bookmarks[:5]:
        print(f"    [L{b['level']}] p{b['physical_page']:>4} | {b['number'] or '—':>6} | {b['title'][:55]}")

# Source B: Sommaire visible
contents_candidates = detect_contents_pages(pages_data)
toc_by_page = extract_contents_entries(pages_data, contents_candidates)
total_toc_entries = sum(len(entries) for entries in toc_by_page.values())

print(f"\n[Source B] Pages de sommaire visible : {len(contents_candidates)} page(s)")
print(f"           Entrées TOC extraites      : {total_toc_entries} entrée(s)")
if toc_by_page:
    first_toc_p = list(toc_by_page.keys())[0]
    print(f"\nExemple Sommaire p{first_toc_p} (5 premières entrées) :")
    for e in toc_by_page[first_toc_p][:5]:
        pg_str = f"→ page imprimée {e['internal_page']}" if e.get('internal_page') else ""
        print(f"    [{e['section_number'] or '—':>6}] {e['title'][:50]:<50} {pg_str}")


[Source A] Bookmarks natifs : 0 entrée(s)



[Source B] Pages de sommaire visible : 46 page(s)
           Entrées TOC extraites      : 1009 entrée(s)

Exemple Sommaire p2 (5 premières entrées) :
    [  1068] -                                                  
    [     2] of                                                 
    [     —] 455A-REF-PRD                                       → page imprimée 9


---
## 🧱 CELL 5 — Build Page Representations (Native Text First — No Massive OCR)

Étape légère : construction des représentations à partir du texte natif et du layout disponible.
**Aucun OCR massif n'est exécuté ici.** Toutes les pages SCANNED restent intactes dans `pages_data`.


In [5]:
# =========================================================================
# CELL 5 — Page Representations (Native Text First — No Massive OCR)
# =========================================================================

# Les représentations initiales sont construites instantanément à partir du texte natif.
# Aucune page SCANNED n'est OCRisée ici. Les pages SCANNED sont conservées à 100%.

print(f"[OK] Représentations de pages natives prêtes pour {len(pages_data)} pages")
print(f"     Pages SCANNED conservées : {scanned_count} (OCR différé et ciblé uniquement)")

# Aperçu de la représentation d'une page
sample_page = pages_data[min(len(pages_data)-1, 5)]
print(f"\nExemple Représentation Page {sample_page['physical_page']} :")
print(f"  - page_type      : {sample_page['page_type']}")
print(f"  - char_count     : {sample_page['char_count']}")
print(f"  - has_native_text: {sample_page['has_native_text']}")
print(f"  - ocr_used       : {sample_page['ocr_used']}")
print(f"  - blocks count   : {len(sample_page['blocks'])}")


[OK] Représentations de pages natives prêtes pour 1068 pages
     Pages SCANNED conservées : 167 (OCR différé et ciblé uniquement)

Exemple Représentation Page 6 :
  - page_type      : TEXT
  - char_count     : 1293
  - has_native_text: True
  - ocr_used       : False
  - blocks count   : 44


---
## 🔍 CELL 6 — Candidate Headings Extraction (Texte Natif)

Détection des candidats de headings sur chaque page texte native (Source C & D) sans recourir à l'OCR.


In [6]:
# =========================================================================
# CELL 6 — Candidate Headings Extraction (Native Text)
# =========================================================================

from document_structure import detect_heading_candidates

contents_pages_set = {c["physical_page"] for c in contents_candidates}
native_headings_by_page = detect_heading_candidates(
    pages_data,
    ocr_blocks_by_page=None,
    contents_page_numbers=contents_pages_set
)

total_cands = sum(len(v) for v in native_headings_by_page.values())
num_cands   = sum(sum(1 for h in v if h.get("section_number")) for v in native_headings_by_page.values())

print(f"[OK] {total_cands} candidats natifs détectés sur {len(native_headings_by_page)} pages TEXT")
print(f"     - Candidats numérotés     : {num_cands}")
print(f"     - Candidats non numérotés : {total_cands - num_cands}")

print(f"\nExemples de candidats natifs bruts (5 premiers) :")
count = 0
for pno in sorted(native_headings_by_page.keys()):
    for h in native_headings_by_page[pno]:
        num_str = h['section_number'] or '—'
        print(f"  p{pno:>4} | {num_str:>8} | score_layout={h['layout_score']:.2f} | {h['raw_text'][:55]}")
        count += 1
        if count >= 5:
            break
    if count >= 5:
        break


[OK] 5527 candidats natifs détectés sur 735 pages TEXT
     - Candidats numérotés     : 826
     - Candidats non numérotés : 4701

Exemples de candidats natifs bruts (5 premiers) :
  p   1 |        — | score_layout=0.60 | LB
  p   1 |        — | score_layout=0.85 | OPERATING AND MAINTENANCE DATA
  p   1 |        — | score_layout=0.95 | LOCATION:
  p   1 |        — | score_layout=0.95 | N°2 AMMONIA STORAGE TANKS
  p   1 |        — | score_layout=0.95 | PROJ.N°:


---
## 📊 CELL 7 — Multi-Signal Heading Scoring

Configuration des poids multi-signaux pour le scoring des candidats.


In [7]:
# =========================================================================
# CELL 7 — Heading Scoring Configuration
# =========================================================================

from document_structure.anchors import DEFAULT_HEADING_WEIGHTS

print("Poids configurés pour le calcul multi-signal du heading_score :")
for sig, w in DEFAULT_HEADING_WEIGHTS.items():
    print(f"  - {sig:<15} : {w:>4.2f}")


Poids configurés pour le calcul multi-signal du heading_score :
  - layout          : 0.30
  - font            : 0.15
  - bold            : 0.10
  - numbering       : 0.30
  - position        : 0.15
  - whitespace      : 0.10
  - toc             : 0.25
  - bookmark        : 0.30
  - style           : 0.10
  - context         : 0.10
  - penalty         : 0.45


---
## 🛡️ CELL 8 — Heading Validation (Texte Natif)

Validation stricte des candidats natifs pour éliminer les faux positifs (stopwords, phrases, tableaux).


In [8]:
# =========================================================================
# CELL 8 — Heading Validation (Native Text)
# =========================================================================

from document_structure import score_and_validate_headings

native_anchors, native_debug_records = score_and_validate_headings(
    headings_by_page=native_headings_by_page,
    pages_data=pages_data,
    toc_by_page=toc_by_page,
    bookmarks=bookmarks,
    min_confidence=MIN_HEADING_CONFIDENCE,
)

accepted_native = [d for d in native_debug_records if d.get("status") == "accepted"]
rejected_native = [d for d in native_debug_records if d.get("status") == "rejected"]

print(f"[OK] Validation initiale sur texte natif :")
print(f"     Total candidats évalués : {len(native_debug_records)}")
print(f"     Acceptés comme anchors  : {len(accepted_native)}")
print(f"     Rejetés (faux headings) : {len(rejected_native)}")


[OK] Validation initiale sur texte natif :
     Total candidats évalués : 5527
     Acceptés comme anchors  : 2195
     Rejetés (faux headings) : 3332


---
## ⚓ CELL 9 — Build Native Section Anchors

Construction des points d'ancrage natifs (`A001`, `A002`, ...).


In [9]:
# =========================================================================
# CELL 9 — Build Native Section Anchors
# =========================================================================

print(f"[OK] {len(native_anchors)} section anchors natifs validés")
print(f"\nAperçu des 10 premiers Anchors Natifs :")
print(f"  {'Anchor ID':<10} {'Page':>6} {'Numéro':>8} {'Confiance':>10}  {'Sources':<20} Titre")
print(f"  {'-'*10} {'-'*6} {'-'*8} {'-'*10}  {'-'*20} {'-'*35}")
for a in native_anchors[:10]:
    src_str = '+'.join(a.get('sources', []))
    num_str = a.get('number') or '—'
    print(f"  {a['anchor_id']:<10} {a['page']:>6} {num_str:>8} {a['confidence']:>10.2f}  {src_str:<20} {a['title'][:35]}")


[OK] 550 section anchors natifs validés

Aperçu des 10 premiers Anchors Natifs :
  Anchor ID    Page   Numéro  Confiance  Sources              Titre
  ---------- ------ -------- ----------  -------------------- -----------------------------------
  A0001           5        —       0.81  native_layout+visible_toc ELECTRICAL
  A0002           7        1       0.99  native_layout+visible_toc GENERAL DOCUMENTS
  A0003          10        —       0.85  native_layout+visible_toc NOISE DATA SHEET
  A0004          11       19       0.99  native_layout+visible_toc LOAD CONDITIONS
  A0005          16        2       0.99  native_layout+visible_toc PACKAGE DRAWINGS
  A0006          19        3       0.99  native_layout+visible_toc P.V. & H.E
  A0007          39        —       0.99  native_layout+visible_toc TEMPERATURE
  A0008          44        —       0.99  native_layout+visible_toc INSPECTION AUTHORITY
  A0009          50        —       0.99  native_layout+visible_toc TEMPERATURE
  A0010        

---
## 🎯 CELL 10 — Identify OCR Required Pages (Analyse d'Incertitude)

Identification précise et déterministe des pages SCANNED où un heading est réellement susceptible d'exister :
- **`OCR_REQUIRED` (HIGH)** : Cibles de TOC/Signets non résolues, trous dans la séquence des sections.
- **`OCR_OPTIONAL` (MEDIUM)** : Pages de transition ou précédant immédiatement un anchor.
- **`OCR_NOT_NEEDED` (LOW)** : Pages scannées au milieu d'une région déjà bien délimitée (aucun OCR exécuté).


In [10]:
# =========================================================================
# CELL 10 — Identify OCR Required Pages (Uncertainty Analysis)
# =========================================================================

from document_structure import (
    identify_ocr_required_pages,
    detect_document_boundaries,
    build_document_segments,
)

boundaries   = detect_document_boundaries(pages_data)
doc_segments = build_document_segments(boundaries, total_pages)

ocr_plan = identify_ocr_required_pages(
    pages_data=pages_data,
    toc_by_page=toc_by_page,
    bookmarks=bookmarks,
    native_anchors=native_anchors,
    doc_segments=doc_segments,
    total_pages=total_pages,
    local_window=OCR_LOCAL_WINDOW,
    expanded_window=OCR_EXPANDED_WINDOW,
)

summary = ocr_plan["summary"]
print(f"=========================================================")
print(f"  ANALYSE DES ZONES D'INCERTITUDE & PAGES SCANNÉES")
print(f"=========================================================")
print(f"  Total pages scannées      : {summary['total_scanned']}")
print(f"  OCR_REQUIRED (Priorité H) : {summary['required_count']} pages")
print(f"  OCR_OPTIONAL (Priorité M) : {summary['optional_count']} pages")
print(f"  OCR_NOT_NEEDED (Évitées)  : {summary['not_needed_count']} pages")
print(f"=========================================================")

if ocr_plan["required"]:
    print(f"\nPages OCR_REQUIRED identifiées :")
    for r in ocr_plan["required"][:8]:
        reasons_str = ', '.join(r['reasons'])
        exp_str = f" [Attendu: {r['expected_sections'][0]}]" if r.get('expected_sections') else ""
        print(f"  - Page {r['physical_page']:>4} ({r['priority']}) : {reasons_str}{exp_str}")


  ANALYSE DES ZONES D'INCERTITUDE & PAGES SCANNÉES
  Total pages scannées      : 167
  OCR_REQUIRED (Priorité H) : 5 pages
  OCR_OPTIONAL (Priorité M) : 18 pages
  OCR_NOT_NEEDED (Évitées)  : 144 pages

Pages OCR_REQUIRED identifiées :
  - Page   86 (HIGH) : near_expected_toc_section('Fig./Abb', target_p=84) [Attendu: Fig./Abb]
  - Page   94 (HIGH) : near_expected_toc_section('ASTM A350 Gr. LF2 Cl', target_p=93), near_expected_toc_section('ASTM A350 Gr. LF2 Cl', target_p=97), near_expected_toc_section('ASME VIII Div', target_p=93), near_expected_toc_section('ASME VIII Div', target_p=97), near_expected_toc_section('455A- REF-LST', target_p=99), near_expected_toc_section('455A-REF-DSS', target_p=93), near_expected_toc_section('455A-REF-DSS', target_p=97), near_expected_toc_section('Fig. 1/Abb', target_p=93), near_expected_toc_section('Fig. 1/Abb', target_p=97), near_expected_toc_section('Fig. 2 Abb', target_p=98), near_expected_toc_section('Fig.3/Abb', target_p=99), near_expected_toc_sec

---
## ⚡ CELL 11 — Targeted Parallel OCR (Two-Level & Caching)

Exécution de l'OCR parallèle uniquement sur les pages nécessaires :
- **Niveau 1 : Light OCR (Top 30%)** pour capture rapide des titres.
- **Niveau 2 : Full OCR** en escalade automatique si un candidat heading est présent.
- **Mise en cache** en mémoire et sur disque (`.ocr_cache/`).


In [11]:
# =========================================================================
# CELL 11 — Targeted Parallel OCR (Two-Level & Caching)
# =========================================================================

from document_structure import run_targeted_ocr_parallel

pages_to_ocr = list(ocr_plan["required"])
if not OCR_ONLY_WHEN_NEEDED:
    pages_to_ocr.extend(ocr_plan["optional"])
elif len(native_anchors) < 5 and ocr_plan["optional"]:
    pages_to_ocr.extend(ocr_plan["optional"])

ocr_results = run_targeted_ocr_parallel(
    pdf_path=INPUT_PDF,
    pages_data=pages_data,
    pages_to_ocr=pages_to_ocr,
    max_workers=MAX_OCR_WORKERS,
    ocr_light_mode=OCR_LIGHT_MODE,
    ocr_top_ratio=OCR_TOP_RATIO,
    ocr_full_on_candidate=OCR_FULL_ON_HEADING_CANDIDATE,
    cache_enabled=OCR_CACHE_ENABLED,
    cache_dir=OCR_CACHE_DIR,
)

ocr_blocks_by_page = ocr_results["ocr_blocks_by_page"]
ocr_metrics = ocr_results["metrics"]

print(f"[OK] OCR Ciblé Parallèle terminé en {ocr_metrics['total_time_sec']}s")
print(f"     Pages traitées          : {ocr_metrics['pages_ocred']}")
print(f"     Light OCR (Top {int(OCR_TOP_RATIO*100)}%)    : {ocr_metrics['light_ocr_count']}")
print(f"     Full OCR                : {ocr_metrics['full_ocr_count']}")
print(f"     Cache Hits              : {ocr_metrics['cache_hits']}")
print(f"     Cache Misses            : {ocr_metrics['cache_misses']}")
print(f"     Temps moyen / page      : {ocr_metrics['avg_time_per_page']}s")
print(f"     Workers utilisés        : {ocr_metrics['max_workers']}")


[OK] OCR Ciblé Parallèle terminé en 0.38s
     Pages traitées          : 5
     Light OCR (Top 30%)    : 5
     Full OCR                : 0
     Cache Hits              : 5
     Cache Misses            : 0
     Temps moyen / page      : 0.076s
     Workers utilisés        : 4


---
## 🔄 CELL 12 — Re-evaluate Headings & Final Anchor Validation

Ré-évaluation des candidats enrichis par l'OCR ciblé et validation finale des anchors.


In [12]:
# =========================================================================
# CELL 12 — Re-evaluate Headings & Final Anchor Validation
# =========================================================================

headings_by_page = detect_heading_candidates(
    pages_data=pages_data,
    ocr_blocks_by_page=ocr_blocks_by_page,
    contents_page_numbers=contents_pages_set
)

anchors, debug_records = score_and_validate_headings(
    headings_by_page=headings_by_page,
    pages_data=pages_data,
    toc_by_page=toc_by_page,
    bookmarks=bookmarks,
    min_confidence=MIN_HEADING_CONFIDENCE,
)

print(f"[OK] Validation finale des anchors : {len(anchors)} anchors confirmés")
print(f"     Anchors natifs          : {len(native_anchors)}")
print(f"     Anchors après OCR ciblé : {len(anchors)}")


[OK] Validation finale des anchors : 550 anchors confirmés
     Anchors natifs          : 550
     Anchors après OCR ciblé : 550


---
## 🌲 CELL 13 — Hierarchy Reconstruction & Document Boundaries

Détection des sous-documents concaténés (`DOC001`, `DOC002`...) et attribution des anchors validés.


In [13]:
# =========================================================================
# CELL 13 — Hierarchy Reconstruction & Document Boundaries
# =========================================================================

from document_structure import anchors_to_documents

structured_documents = anchors_to_documents(anchors, doc_segments)

print(f"[OK] {len(doc_segments)} sous-documents structurés")
for seg in doc_segments[:5]:
    n_p = seg['end_page'] - seg['start_page'] + 1
    print(f"  - {seg['document_id']} : pages {seg['start_page']:>4} à {seg['end_page']:>4} ({n_p:>4} pages) | {seg.get('evidence','')[:50]}")


[OK] 39 sous-documents structurés
  - DOC001 : pages    1 à    8 (   8 pages) | First page of PDF | Contents header (DOCUMENT INDE
  - DOC002 : pages    9 à   12 (   4 pages) | Contents header (DOCUMENT INDEX)
  - DOC003 : pages   13 à   20 (   8 pages) | Contents header (DOCUMENT INDEX)
  - DOC004 : pages   21 à   22 (   2 pages) | Contents header (DOCUMENT INDEX)
  - DOC005 : pages   23 à   30 (   8 pages) | Frontsheet title (DOCUMENT FRONTSHEET)


---
## 📏 CELL 14 — Section Boundaries Construction (`start_page` / `end_page`)

Construction des frontières exactes de chaque section avec gestion du preamble et emboîtement L1/L2/L3.


In [14]:
# =========================================================================
# CELL 14 — Section Boundaries Construction
# =========================================================================

from document_structure import build_section_boundaries

sections = build_section_boundaries(structured_documents, total_pages)

levels = {}
for s in sections:
    lvl = s.get("level", 1)
    levels[lvl] = levels.get(lvl, 0) + 1

print(f"[OK] {len(sections)} sections définitives construites")
print(f"\nRépartition hiérarchique :")
for lvl in sorted(levels.keys()):
    print(f"  Niveau L{lvl} : {levels[lvl]:>4} sections")

print(f"\nAperçu des 15 premières sections :")
print(f"  {'ID':<6} {'L':>2} {'Début':>6} {'Fin':>6} {'Pages':>6}  {'Numéro':>8}  Titre")
print(f"  {'-'*6} {'-'*2} {'-'*6} {'-'*6} {'-'*6}  {'-'*8}  {'-'*40}")
for s in sections[:15]:
    indent = "  " * (s.get("level", 1) - 1)
    n_pg = s["end_page"] - s["start_page"] + 1
    num_str = s.get("number") or "—"
    print(f"  {s['id']:<6} L{s.get('level',1):>1} {s['start_page']:>6} {s['end_page']:>6} {n_pg:>6}  {num_str:>8}  {indent}{s['title'][:40]}")


[OK] 589 sections définitives construites

Répartition hiérarchique :
  Niveau L1 :  387 sections
  Niveau L2 :  148 sections
  Niveau L3 :   46 sections
  Niveau L4 :    8 sections

Aperçu des 15 premières sections :
  ID      L  Début    Fin  Pages    Numéro  Titre
  ------ -- ------ ------ ------  --------  ----------------------------------------
  S001   L1      1      4      4         —  DOCUMENT PREAMBLE & CONTENTS (DOC001)
  S002   L1      5      6      2         —  ELECTRICAL
  S003   L1      7      8      2         1  GENERAL DOCUMENTS
  S004   L1      9      9      1         —  DOCUMENT PREAMBLE & CONTENTS (DOC002)
  S005   L1     10     10      1         —  NOISE DATA SHEET
  S006   L1     11     12      2        19  LOAD CONDITIONS
  S007   L1     13     15      3         —  DOCUMENT PREAMBLE & CONTENTS (DOC003)
  S008   L1     16     18      3         2  PACKAGE DRAWINGS
  S009   L1     19     20      2         3  P.V. & H.E
  S010   L1     21     22      2         —  DOC

---
## 🗺️ CELL 15 — 100% Page Assignment & Continuity Mapping

Attribution de **100 % des pages physiques** (1 à N) à leur section respective par le Principe de Continuité.
Toutes les pages scannées sont intégrées dans le `page_map`.


In [15]:
# =========================================================================
# CELL 15 — 100% Page Assignment & Continuity Mapping
# =========================================================================

from document_structure import assign_all_pages_to_sections

page_map = assign_all_pages_to_sections(sections, pages_data, total_pages)

assigned_count   = sum(1 for p in page_map if p.get("section_id") not in (None, "UNASSIGNED"))
unassigned_count = total_pages - assigned_count

method_counts = {}
for p in page_map:
    m = p.get("assignment_method", "unknown")
    method_counts[m] = method_counts.get(m, 0) + 1

print(f"[OK] Attribution terminée pour {len(page_map)} / {total_pages} pages")
print(f"     Pages assignées     : {assigned_count} (100.0%)")
print(f"     Pages non assignées : {unassigned_count}")
print(f"\nMéthodes d'assignation :")
for m, c in sorted(method_counts.items()):
    print(f"  - {m:<30} : {c:>5} pages")

print(f"\nAperçu Page Map (10 premières pages) :")
for p in page_map[:10]:
    print(f"  Page {p['page']:>4} → Section {p['section_id']:<6} [{p['page_type']:<7}] (méthode: {p['assignment_method']}, conf: {p.get('assignment_confidence')})")


[OK] Attribution terminée pour 1068 / 1068 pages
     Pages assignées     : 1068 (100.0%)
     Pages non assignées : 0

Méthodes d'assignation :
  - anchor                         :   503 pages
  - continuity                     :   565 pages

Aperçu Page Map (10 premières pages) :
  Page    1 → Section S001   [TEXT   ] (méthode: anchor, conf: 1.0)
  Page    2 → Section S001   [TEXT   ] (méthode: continuity, conf: 0.98)
  Page    3 → Section S001   [TEXT   ] (méthode: continuity, conf: 0.98)
  Page    4 → Section S001   [TEXT   ] (méthode: continuity, conf: 0.98)
  Page    5 → Section S002   [TEXT   ] (méthode: anchor, conf: 0.81)
  Page    6 → Section S002   [TEXT   ] (méthode: continuity, conf: 0.98)
  Page    7 → Section S003   [TEXT   ] (méthode: anchor, conf: 0.99)
  Page    8 → Section S003   [SCANNED] (méthode: continuity, conf: 0.98)
  Page    9 → Section S004   [TEXT   ] (méthode: anchor, conf: 1.0)
  Page   10 → Section S005   [TEXT   ] (méthode: anchor, conf: 0.85)


---
## ✅ CELL 16 — Automated Structural Integrity Validation (10 Rules)

Vérification des 10 règles obligatoires de validation structurelle.


In [16]:
# =========================================================================
# CELL 16 — Automated Structural Integrity Validation (10 Rules)
# =========================================================================

from document_structure import validate_structural_integrity

validation = validate_structural_integrity(
    sections=sections,
    page_map=page_map,
    doc_segments=doc_segments,
    total_pages=total_pages,
)

status = "VALIDE" if validation["valid"] else "INVALIDE"
print(f"=========================================================")
print(f"  RÉSULTAT DE LA VALIDATION : {status}")
print(f"  Couverture globale        : {validation['coverage']*100:.2f}%")
print(f"  Pages non assignées       : {validation['unassigned_pages']}")
print(f"  Erreurs structurelles     : {len(validation['errors'])}")
print(f"  Avertissements            : {len(validation['warnings'])}")
print(f"=========================================================")

for r in validation.get("rule_results", []):
    icon = "✅ [OK]" if r["passed"] else "❌ [ERR]"
    print(f"  {icon} {r['rule_id']} — {r['description']}")


  RÉSULTAT DE LA VALIDATION : VALIDE
  Couverture globale        : 100.00%
  Pages non assignées       : 0
  Erreurs structurelles     : 0
  Avertissements            : 0
  ✅ [OK] V01 — Toutes les sections ont start_page <= end_page
  ✅ [OK] V02 — start_page >= 1 et end_page <= total_pages
  ✅ [OK] V03 — Sections ordonnées de manière monotone
  ✅ [OK] V04 — Aucune section ne traverse une frontière de sous-document
  ✅ [OK] V05 — Les sections enfants sont strictement contenues dans leur parent
  ✅ [OK] V06 — Les sections de même niveau ne se chevauchent pas
  ✅ [OK] V07 — Couverture de pages = 100% (100.0%)
  ✅ [OK] V08 — Zéro page non assignée (0 trouvées)
  ✅ [OK] V09 — IDs de sections uniques
  ✅ [OK] V10 — Toutes les pages référencent une section existante


---
## 📊 CELL 17 — OCR Diagnostics & Performance Benchmark

Rapport complet de diagnostic et de performance comparant l'ancien comportement théorique avec le nouvel OCR ciblé.


In [17]:
# =========================================================================
# CELL 17 — OCR Diagnostics & Performance Benchmark
# =========================================================================

from document_structure import generate_quality_report

estimated_old_ocr_pages = scanned_count
actual_ocr_pages        = ocr_metrics["pages_ocred"]
reduction_percent       = ((estimated_old_ocr_pages - actual_ocr_pages) / max(1, estimated_old_ocr_pages)) * 100.0
saved_pages             = estimated_old_ocr_pages - actual_ocr_pages

print("=========================================================")
print("  📊 BILAN DE PERFORMANCE & DIAGNOSTIC OCR CIBLÉ")
print("=========================================================")
print(f"  Total pages document    : {total_pages}")
print(f"  Pages TEXT              : {text_count}")
print(f"  Pages SCANNED totales   : {scanned_count}")
print(f"---------------------------------------------------------")
print(f"  Pages SCANNED OCRisées  : {actual_ocr_pages}")
print(f"  Pages SCANNED évitées   : {saved_pages}")
print(f"  Light OCR (Top {int(OCR_TOP_RATIO*100)}%)    : {ocr_metrics['light_ocr_count']}")
print(f"  Full OCR                : {ocr_metrics['full_ocr_count']}")
print(f"  Cache Hits              : {ocr_metrics['cache_hits']}")
print(f"  Cache Misses            : {ocr_metrics['cache_misses']}")
print(f"  Temps OCR total         : {ocr_metrics['total_time_sec']}s")
print(f"  Temps moyen / page      : {ocr_metrics['avg_time_per_page']}s")
print(f"  Nombre de workers       : {ocr_metrics['max_workers']}")
print(f"---------------------------------------------------------")
print(f"  Ancien OCR (toutes SC)  : {estimated_old_ocr_pages} pages")
print(f"  Nouveau OCR (ciblé)     : {actual_ocr_pages} pages")
print(f"  🎯 Réduction de calcul  : {reduction_percent:.1f}%")
print("=========================================================\n")

report = generate_quality_report(
    filename       = INPUT_PDF.name,
    total_pages    = total_pages,
    doc_segments   = doc_segments,
    sections       = sections,
    pages_data     = pages_data,
    page_map       = page_map,
    validation_res = validation,
    ocr_pages_count= actual_ocr_pages,
)
print(report)


  📊 BILAN DE PERFORMANCE & DIAGNOSTIC OCR CIBLÉ
  Total pages document    : 1068
  Pages TEXT              : 821
  Pages SCANNED totales   : 167
---------------------------------------------------------
  Pages SCANNED OCRisées  : 5
  Pages SCANNED évitées   : 162
  Light OCR (Top 30%)    : 5
  Full OCR                : 0
  Cache Hits              : 5
  Cache Misses            : 0
  Temps OCR total         : 0.38s
  Temps moyen / page      : 0.076s
  Nombre de workers       : 4
---------------------------------------------------------
  Ancien OCR (toutes SC)  : 167 pages
  Nouveau OCR (ciblé)     : 5 pages
  🎯 Réduction de calcul  : 97.0%

DOCUMENT DOC001 (MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf)
Pages:
    8
Sections:
    3
TEXT:
    7
SCANNED:
    1
MIXED:
    0
Coverage:
    100.0%
Unassigned:
    0
----------------------------------------
SECTIONS:
  S001   DOCUMENT PREAMBLE & CONTENTS (DOC001)                   1-4  (conf=1.00)
  S002   ELECTRICAL       

---
## 💾 CELL 18 — Save `structure.json`

Export du fichier structure standardisé destiné au LLM pour la sélection des régions pertinentes.


In [18]:
# =========================================================================
# CELL 18 — Save structure.json
# =========================================================================

import json

structure_data = {
    "document": {
        "filename": INPUT_PDF.name,
        "total_pages": total_pages
    },
    "sections": [
        {
            "id": s["id"],
            "number": s.get("number"),
            "title": s["title"],
            "level": s["level"],
            "start_page": s["start_page"],
            "end_page": s["end_page"],
            "confidence": s.get("confidence", 0.95)
        }
        for s in sections
    ]
}

structure_path = OUTPUT_DIR / "structure.json"
with open(structure_path, "w", encoding="utf-8") as f:
    json.dump(structure_data, f, ensure_ascii=False, indent=2)

print(f"[OK] structure.json sauvegardé : {structure_path.resolve()}")
print(f"     {len(structure_data['sections'])} sections exportées")
print(f"\nExtrait (5 premières sections) :")
print(json.dumps(structure_data["sections"][:5], indent=2, ensure_ascii=False))


[OK] structure.json sauvegardé : C:\Users\camelia\Downloads\projets\projet1OCP\output\structure.json
     589 sections exportées

Extrait (5 premières sections) :
[
  {
    "id": "S001",
    "number": null,
    "title": "DOCUMENT PREAMBLE & CONTENTS (DOC001)",
    "level": 1,
    "start_page": 1,
    "end_page": 4,
    "confidence": 1.0
  },
  {
    "id": "S002",
    "number": null,
    "title": "ELECTRICAL",
    "level": 1,
    "start_page": 5,
    "end_page": 6,
    "confidence": 0.81
  },
  {
    "id": "S003",
    "number": "1",
    "title": "GENERAL DOCUMENTS",
    "level": 1,
    "start_page": 7,
    "end_page": 8,
    "confidence": 0.99
  },
  {
    "id": "S004",
    "number": null,
    "title": "DOCUMENT PREAMBLE & CONTENTS (DOC002)",
    "level": 1,
    "start_page": 9,
    "end_page": 9,
    "confidence": 1.0
  },
  {
    "id": "S005",
    "number": null,
    "title": "NOISE DATA SHEET",
    "level": 1,
    "start_page": 10,
    "end_page": 10,
    "confidence": 0.855
  }
]


---
## 🗺️ CELL 19 — Save `page_map.json`

Export de la table de correspondance complète page physique (1..N) → section.


In [19]:
# =========================================================================
# CELL 19 — Save page_map.json
# =========================================================================

page_map_data = {
    "pages": [
        {
            "page": p["page"],
            "section_id": p["section_id"],
            "page_type": p["page_type"],
            "assignment_method": p["assignment_method"],
            "assignment_confidence": p.get("assignment_confidence", 0.95)
        }
        for p in page_map
    ]
}

page_map_path = OUTPUT_DIR / "page_map.json"
with open(page_map_path, "w", encoding="utf-8") as f:
    json.dump(page_map_data, f, ensure_ascii=False, indent=2)

print(f"[OK] page_map.json sauvegardé : {page_map_path.resolve()}")
print(f"     {len(page_map_data['pages'])} pages mappées à 100%")
print(f"\nExtrait (5 premières pages) :")
print(json.dumps(page_map_data["pages"][:5], indent=2, ensure_ascii=False))


[OK] page_map.json sauvegardé : C:\Users\camelia\Downloads\projets\projet1OCP\output\page_map.json
     1068 pages mappées à 100%

Extrait (5 premières pages) :
[
  {
    "page": 1,
    "section_id": "S001",
    "page_type": "TEXT",
    "assignment_method": "anchor",
    "assignment_confidence": 1.0
  },
  {
    "page": 2,
    "section_id": "S001",
    "page_type": "TEXT",
    "assignment_method": "continuity",
    "assignment_confidence": 0.98
  },
  {
    "page": 3,
    "section_id": "S001",
    "page_type": "TEXT",
    "assignment_method": "continuity",
    "assignment_confidence": 0.98
  },
  {
    "page": 4,
    "section_id": "S001",
    "page_type": "TEXT",
    "assignment_method": "continuity",
    "assignment_confidence": 0.98
  },
  {
    "page": 5,
    "section_id": "S002",
    "page_type": "TEXT",
    "assignment_method": "anchor",
    "assignment_confidence": 0.81
  }
]


---
## 🔍 CELL 20 — Save `structure_debug.json`

Export du journal d'audit complet (candidats, frontières, métriques OCR et validation).


In [20]:
# =========================================================================
# CELL 20 — Save structure_debug.json
# =========================================================================

debug_data = {
    "document": {
        "filename": INPUT_PDF.name,
        "total_pages": total_pages,
        "segments": doc_segments
    },
    "sections": [
        {
            "id": s["id"],
            "number": s.get("number"),
            "title": s["title"],
            "level": s["level"],
            "start_page": s["start_page"],
            "end_page": s["end_page"],
            "confidence": s.get("confidence", 0.95),
            "anchor": s.get("anchor"),
            "debug": s.get("debug", {})
        }
        for s in sections
    ],
    "ocr_metrics": ocr_metrics,
    "ocr_plan": ocr_plan["summary"],
    "candidates": debug_records,
    "validation": validation
}

debug_path = OUTPUT_DIR / "structure_debug.json"
with open(debug_path, "w", encoding="utf-8") as f:
    json.dump(debug_data, f, ensure_ascii=False, indent=2)

print(f"[OK] structure_debug.json sauvegardé : {debug_path.resolve()}")
print(f"     Artefacts complets générés dans : {OUTPUT_DIR.resolve()}")


[OK] structure_debug.json sauvegardé : C:\Users\camelia\Downloads\projets\projet1OCP\output\structure_debug.json
     Artefacts complets générés dans : C:\Users\camelia\Downloads\projets\projet1OCP\output
